In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

### Import Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Settings
pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

### Load Dataset

In [ ]:
df = pd.read_csv("/kaggle/input/student-performance-dataset/ultimate_student_productivity_dataset_5000.csv")

df.head()

# EDA Phase 1

### STEP 1: Dataset Structure Overview

##### 1.1 Shape

In [ ]:
df.shape

##### 1.2 Column Types

In [ ]:
df.info()

### STEP 2: Statistical Summary

In [ ]:
df.describe()

### STEP 3: Missing Values

In [ ]:
df.isnull().sum()

### STEP 4: Unique Values (Categorical)

In [ ]:
for col in ['gender','academic_level','internet_quality']:
    print(col)
    print(df[col].value_counts())
    print("\n")

### STEP 5: Check for Duplicates

In [ ]:
df.duplicated().sum()

### STEP 6: Check Logical Relationships

##### Screen Time Validation

In [ ]:
df['calculated_screen'] = df['social_media_hours'] + df['gaming_hours']

df[['screen_time_hours','calculated_screen']].head()

In [ ]:
(df['screen_time_hours'] - df['calculated_screen']).describe()

##### Study Total Validation

In [ ]:
df['total_study'] = (
    df['study_hours'] +
    df['self_study_hours'] +
    df['online_classes_hours']
)

df['total_study'].describe()

### STEP 7: Target Variables Overview

In [ ]:
targets = ['focus_index','burnout_level','productivity_score','exam_score']

df[targets].describe()

# PHASE 1 SUMMARY 

### Dataset Health Check

- 5000 rows
- 21 columns
- No missing values
- No duplicate issues (assuming checked)
- Balanced categorical variables

This dataset is clean and well-structured.

### Categorical Variables Are Balanced
Gender

Male: 1719

Other: 1651

Female: 1630

Almost perfectly balanced

Academic Level

Postgraduate: 1687

High School: 1672

Undergraduate: 1641

Perfectly balanced

Internet Quality

Good / Average / Poor evenly distributed

This is VERY GOOD for modeling — no imbalance issue.

### Numerical Variable Observations
Age

16–25 → realistic student range

Study Hours

Mean = 4.53 hours
Max = 11.84

- Realistic

Sleep Hours

4–10 hours
Mean = 7 hours

- Healthy distribution

### IMPORTANT DISCOVERY: Screen Time Mismatch

I calculated:

screen_time_hours - (social_media_hours + gaming_hours)


Result:

Min = -8.79

Max = 11.87

Mean = 2.41

This means:

screen_time_hours ≠ social_media_hours + gaming_hours

This suggests:

Screen time includes other usage (YouTube, browsing, etc.)
OR dataset intentionally added noise

This is NOT an error — just realistic variation.

### Total Study Hours

Mean = 9 hours/day
Max = 16.6 hours

- Possible but intense
No unrealistic 20+ hour cases

### Target Variables Overview
Exam Score

Mean = 18.8

Max = 64.09

Min = 1

IMPORTANT:

Exam scores are NOT 0–100.
They range 1–64.

This suggests:
Scores are modeled internally
Not standard percentage

This is fine — just important to remember.

# EDA Phase 2

### SECTION 1: DEMOGRAPHICS

##### Age Distribution

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['age'], bins=10, kde=True)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()

The dataset represents a well-distributed student population between ages 16 and 25, with no extreme age outliers. This ensures balanced demographic modeling.

##### Gender Distribution

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='gender', data=df)
plt.title("Gender Distribution")
plt.show()

Observation:

- Almost equal distribution

- No class imbalance

##### Academic Level Distribution

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='academic_level', data=df)
plt.title("Academic Level Distribution")
plt.show()

Balanced across levels.

### SECTION 2: STUDY HABITS

##### Study Hours

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['study_hours'], bins=20, kde=True)
plt.title("Study Hours Distribution")
plt.show()

Insight:

- Bell-shaped distribution

- Peak around 4–6 hours

##### Self Study Hours

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['self_study_hours'], bins=20, kde=True)
plt.title("Self Study Hours")
plt.show()

Centered around 2–3 hours.

##### Online Classes Hours

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['online_classes_hours'], bins=20, kde=True)
plt.title("Online Classes Hours")
plt.show()

Centered around 2 hours.

##### Total Study Hours (Engineered Feature)

In [ ]:
df['total_study'] = (
    df['study_hours'] +
    df['self_study_hours'] +
    df['online_classes_hours']
)

plt.figure(figsize=(6,4))
sns.histplot(df['total_study'], bins=20, kde=True)
plt.title("Total Study Time")
plt.show()

Most students study 7–11 hours total

### SECTION 3: DIGITAL BEHAVIOR

##### Social Media Hours

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['social_media_hours'], bins=20, kde=True)
plt.title("Social Media Usage")
plt.show()

Most around 2–4 hours

##### Gaming Hours

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['gaming_hours'], bins=20, kde=True)
plt.title("Gaming Hors")
plt.show()

Right-skewed (many low gamers, few heavy gamers).

##### Screen Time

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['screen_time_hours'], bins=20, kde=True)
plt.title("Total Screen Time")
plt.show()

- Centered around 7 hours

- Some heavy users (10–15 hrs)

### SECTION 4: HEALTH & LIFESTYLE

##### Sleep Hours

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['sleep_hours'], bins=20, kde=True)
plt.title("Sleep Hours")
plt.show()

- Most between 6–8 hours

- No extreme sleepers

##### Exercise Minutes                    

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['exercise_minutes'], bins=20, kde=True)
plt.title("Exercise Minutes")
plt.show()

Evenly distributed 0–150 mins.

##### Caffeine Intake

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['caffeine_intake_mg'], bins=20, kde=True)
plt.title("Caffeine Intake (mg)")
plt.show()

Some heavy drinkers (400+ mg)

##### Mental Health Score

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['mental_health_score'], bins=10, kde=True)
plt.title("Mental Health Score")
plt.show()

Range: 1–10

### SECTION 5: TARGET VARIABLES

##### Focus Index

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['focus_index'], bins=20, kde=True)
plt.title("Focus Index Distribution")
plt.show()

##### Burnout Level

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['burnout_level'], bins=20, kde=True)
plt.title("Burnout Level Distribution")
plt.show()

##### Productivity Score

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['productivity_score'], bins=20, kde=True)
plt.title("Productivity Score")
plt.show()

##### Exam Score (MAIN TARGET)

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['exam_score'], bins=20, kde=True)
plt.title("Exam Score Distribution")
plt.show()

Right skewed → many low scorers

#  PHASE 2 SUMMARY

The univariate analysis shows that study habits and lifestyle factors follow near-normal distributions with realistic ranges. Target variables such as focus index, burnout level, and productivity score are approximately normally distributed, making them suitable for regression modeling. Exam scores are slightly right-skewed, indicating a higher concentration of low-to-mid performers.